In [12]:
# 0. Install zstd (required for Ollama installation)
!apt-get update && apt-get install -y zstd

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background
import subprocess
import time

# Use subprocess to run 'ollama serve' without blocking the cell
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Give it a few seconds to wake up

# 3. Pull the models you need
!ollama pull llama3
!ollama pull nomic-embed-text

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [9]:
!pip install -U langchain-ollama langchain-chroma langchain-core pypdf langchain-community

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

def ingest_pdf(file_path):
    # 1. Load PDF
    loader = PyPDFLoader(file_path)
    data = loader.load()

    # 2. Split text
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(data)

    # 3. Create Vector DB
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=OllamaEmbeddings(model="nomic-embed-text"),
        persist_directory="../db" # Save to root db folder
    )
    print("Vector database created and saved to /db")

if __name__ == "__main__":
    ingest_pdf("../data/system_analysis.pdf")

Vector database created and saved to /db


### Uploading the PDF File

To resolve the `ValueError`, please upload the `system_analysis.pdf` file to your Colab session. You can do this by:

1.  Clicking the 'Files' icon (folder icon) on the left sidebar.
2.  Clicking the 'Upload to session storage' icon (page with an arrow pointing up).
3.  Selecting `system_analysis.pdf` from your local machine.

In [14]:
import os
import shutil

file_name = "system_analysis.pdf"
source_path = f"/content/system_analysis.pdf"
dest_dir = "../data"
dest_path = f"{dest_dir}/{file_name}"

# Create the target directory if it doesn't exist
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)
    print(f"Created directory: {dest_dir}")

# Check if the file was uploaded to the root /content/ directory
if os.path.exists(source_path):
    shutil.move(source_path, dest_path)
    print(f"Moved {file_name} from {source_path} to {dest_path}")
    # Now call the ingest_pdf function from the previous cell
    ingest_pdf(dest_path)
else:
    print(f"Error: {file_name} not found at {source_path}. Please ensure the file is uploaded.")

Error: system_analysis.pdf not found at /content/system_analysis.pdf. Please ensure the file is uploaded.


In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.llms import Ollama # Corrected import path for Ollama

# Reload the vector_db as it was created in a local scope previously
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
vector_db = Chroma(persist_directory="../db", embedding_function=embedding_model)

# Initialize the LLM
llm = Ollama(model="llama3")

# Defining the chain using the pipe (|) operator
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {"context": vector_db.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Usage:
# chain.invoke("What is the system analysis?")

In [15]:
# Example usage:
response = chain.invoke("What is the system analysis?")
print(response)

According to the context, System Analysis is the very first step in any system development where developers come together to understand the problem, needs, and objectives of the project. Some of its key aspects are:

* Problem Identification: identifying the issues that the system is aiming to address
* Requirements Gathering: gathering and writing down the requirements, involving communication with customers and developers
* Feasibility study: evaluating technical, operational, and financial aspects to determine the feasibility of the proposed solution
